# 🛒 BasketAI — Market Basket Analysis System
### Apriori + FP-Growth + ECLAT | Recommendation Engine | Customer Segmentation
---
**Run cells top to bottom. Every path is resolved automatically — no manual folder navigation needed.**

## ⚙️ Cell 1 — Path Setup (run this FIRST)

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')

def find_project_root():
    """Walk upward from cwd to find the folder containing app.py or ml_engine.py."""
    candidates = [os.getcwd()]
    for base in list(candidates):
        candidates += [os.path.join(base,'..'), os.path.join(base,'..','..') ]
    markers = ['app.py','ml_engine.py', os.path.join('backend','app.py')]
    for c in candidates:
        c = os.path.abspath(c)
        for m in markers:
            if os.path.exists(os.path.join(c,m)):
                return c
    return os.getcwd()

ROOT = find_project_root()

# Add every folder that could contain ml_engine.py
for p in [ROOT, os.path.join(ROOT,'backend'), os.path.join(ROOT,'..'), os.getcwd()]:
    p = os.path.abspath(p)
    if p not in sys.path:
        sys.path.insert(0, p)

# Resolve data directory
DATA_DIR = None
for dc in [os.path.join(ROOT,'data'), os.path.join(ROOT,'..','data'), os.path.join(os.getcwd(),'data')]:
    if os.path.isdir(os.path.abspath(dc)):
        DATA_DIR = os.path.abspath(dc)
        break
if DATA_DIR is None:
    DATA_DIR = os.path.join(ROOT,'data')
    os.makedirs(DATA_DIR, exist_ok=True)

DATASET = os.path.join(DATA_DIR, 'transactions.csv')

# Resolve app.py
APP_PY = next(
    (p for p in [os.path.join(ROOT,'app.py'), os.path.join(ROOT,'backend','app.py')] if os.path.exists(p)),
    os.path.join(ROOT,'app.py')
)

print(f'Project root : {ROOT}')
print(f'Data dir     : {DATA_DIR}')
print(f'Dataset      : {DATASET}  (exists={os.path.exists(DATASET)})')
print(f'app.py       : {APP_PY}   (exists={os.path.exists(APP_PY)})')
print('\n✅ Paths ready — run Cell 2')

## 📦 Cell 2 — Install Dependencies

In [ ]:
import subprocess, importlib

PACKAGES = ['fastapi','uvicorn','python-multipart','pandas','numpy','scikit-learn','matplotlib']

def installed(pkg):
    try: importlib.import_module(pkg.split('[')[0].replace('-','_')); return True
    except ImportError: return False

missing = [p for p in PACKAGES if not installed(p)]
if missing:
    print(f'Installing: {missing} ...')
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet']+missing)
print('✅ All packages ready')

## 🔧 Cell 3 — Import ML Engine

In [ ]:
import pandas as pd, numpy as np

try:
    from ml_engine import (
        load_transactions, run_full_analysis,
        Apriori, FPGrowth, ECLAT,
        RecommendationEngine, rank_rules,
        segment_customers, analyze_seasonality,
        build_product_graph, compare_models
    )
    print('✅ ml_engine imported')
except ModuleNotFoundError as e:
    print(f'❌ {e}')
    print('sys.path searched:')
    for p in sys.path: print(f'  {p}')
    raise

## 📊 Cell 4 — Generate & Load Dataset

In [ ]:
if not os.path.exists(DATASET):
    print('Generating sample data...')
    import random; from datetime import datetime, timedelta
    random.seed(42); np.random.seed(42)
    PRODS = {
        'Electronics':['Laptop','Mouse','Keyboard','Monitor','Webcam','Headphones','USB Hub','Laptop Stand','Charger','Speaker'],
        'Office':['Notebook','Pen Set','Desk Organizer','Stapler','Sticky Notes','File Folder','Whiteboard','Printer Paper','Scissors','Tape'],
        'Beverages':['Coffee','Green Tea','Protein Shake','Energy Drink','Water Bottle','Juice Pack','Herbal Tea','Cold Brew','Kombucha','Coconut Water'],
        'Snacks':['Granola Bar','Mixed Nuts','Dark Chocolate','Popcorn','Trail Mix','Rice Cakes','Protein Bar','Dried Fruit','Crackers','Yogurt'],
        'Health':['Vitamins','Omega-3','Protein Powder','Probiotics','Melatonin','Zinc','Vitamin D','Magnesium','Collagen','Turmeric'],
        'Fitness':['Resistance Band','Yoga Mat','Foam Roller','Jump Rope','Dumbbells','Water Bottle','Gym Gloves','Workout Towel','Ankle Weights','Balance Board'],
    }
    ASSOC = {'Laptop':['Mouse','Keyboard','Laptop Stand','USB Hub'],'Coffee':['Dark Chocolate','Granola Bar'],
             'Protein Shake':['Protein Bar','Resistance Band'],'Yoga Mat':['Foam Roller','Water Bottle'],
             'Vitamins':['Omega-3','Probiotics'],'Headphones':['Laptop','Speaker']}
    SEGS = {'tech_professional':{'cats':['Electronics','Office','Beverages'],'w':0.25},
            'health_enthusiast':{'cats':['Health','Fitness','Beverages','Snacks'],'w':0.30},
            'office_worker':{'cats':['Office','Beverages','Snacks'],'w':0.25},
            'student':{'cats':['Electronics','Office','Snacks','Beverages'],'w':0.20}}
    sn=list(SEGS.keys()); sw=[SEGS[s]['w'] for s in sn]
    start=datetime(2023,1,1); recs=[]
    for tid in range(1,2001):
        date=start+timedelta(days=random.randint(0,364))
        seg=random.choices(sn,weights=sw)[0]
        cid=f'CUST_{random.randint(1,500):04d}'
        basket=list(set([random.choice(PRODS[random.choice(SEGS[seg]['cats'])]) for _ in range(random.randint(2,8))]))
        for p in basket[:]:
            if p in ASSOC and random.random()<0.6: basket.append(random.choice(ASSOC[p]))
        for prod in set(basket):
            recs.append({'transaction_id':f'TXN_{tid:05d}','customer_id':cid,'product':prod,
                         'date':date.strftime('%Y-%m-%d'),'month':date.month,'day_of_week':date.strftime('%A'),
                         'customer_segment':seg,'quantity':random.randint(1,3),'price':round(random.uniform(5,200),2)})
    pd.DataFrame(recs).to_csv(DATASET,index=False)
    print(f'✅ Saved to {DATASET}')

df, basket_list, all_items = load_transactions(DATASET)
print(f'Transactions : {df["transaction_id"].nunique():,}')
print(f'Customers    : {df["customer_id"].nunique():,}')
print(f'Products     : {df["product"].nunique()}')
print(f'Date range   : {df["date"].min()} to {df["date"].max()}')
df.head(10)

## 🤖 Cell 5 — Algorithm Comparison

In [ ]:
params = {'min_support':0.01,'min_confidence':0.20,'max_len':3}
print('Training Apriori, FP-Growth, ECLAT...')
comparison = compare_models(basket_list, params)
rows = [{'Algorithm':a,'Itemsets':d['n_frequent_itemsets'],'Rules':d['n_rules'],
         'Time(s)':d['training_time_sec'],'Avg Lift':d['avg_lift'],'Max Lift':d['max_lift']}
        for a,d in comparison.items()]
pd.DataFrame(rows).style.background_gradient(subset=['Rules','Avg Lift','Max Lift'], cmap='Blues')

## 🔗 Cell 6 — Top Association Rules

In [ ]:
model = FPGrowth(**params)
model.fit(basket_list)
rules = rank_rules(model.rules)
print(f'Found {len(rules)} rules')
if rules:
    rdf = pd.DataFrame(rules)[['antecedents','consequents','support','confidence','lift','interestingness','algorithm']]
    rdf['antecedents'] = rdf['antecedents'].apply(lambda x:' + '.join(x))
    rdf['consequents'] = rdf['consequents'].apply(lambda x:' + '.join(x))
    display(rdf.head(15).style.background_gradient(subset=['lift','confidence','interestingness'], cmap='YlOrRd'))

## 🔮 Cell 7 — Recommendation Engine

In [ ]:
rec_engine = RecommendationEngine()
rec_engine.fit(rules)
for cart in [['Laptop','Mouse'],['Yoga Mat','Water Bottle'],['Coffee','Notebook']]:
    r = rec_engine.recommend(cart, n=3)
    print(f'\n🛒 Cart: {cart}')
    for u in r['upsell']:   print(f'  ⬆️  {u["product"]} (lift={u["lift"]:.2f}) — {u["explanation"]}')
    for c in r['cross_sell']: print(f'  ↔️  {c["product"]} (lift={c["lift"]:.2f})')
    if not r['upsell'] and not r['cross_sell']: print('  ℹ️  No strong associations')

## 👥 Cell 8 — Customer Segmentation

In [ ]:
customer_segs = segment_customers(df)
print(customer_segs['segment'].value_counts())
customer_segs.groupby('segment')[['total_spend','n_transactions','n_unique_products']].mean().round(2)

## 📅 Cell 9 — Seasonality Analysis

In [ ]:
season = analyze_seasonality(df)
tdf    = pd.DataFrame(season['trend_data'])
top3   = season['top_products'][:3]
tdf[['month']+top3].style.background_gradient(subset=top3, cmap='Blues')

## 📊 Cell 10 — Matplotlib Dashboard

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('dark_background')
fig, axes = plt.subplots(2,2,figsize=(14,10))
fig.patch.set_facecolor('#050810')
clrs = ['#3b82f6','#06b6d4','#8b5cf6','#f59e0b','#10b981']

ax=axes[0,0]; ax.set_facecolor('#0c1120')
algos=[a for a in comparison]; nr=[comparison[a]['n_rules'] for a in algos]
bars=ax.bar(algos,nr,color=clrs[:3],edgecolor='none',alpha=0.85)
ax.set_title('Rules per Algorithm',color='#e2e8f0',pad=10)
ax.tick_params(colors='#64748b')
for s in ['top','right','left']: ax.spines[s].set_visible(False)
ax.spines['bottom'].set_color('#1e293b')
for b,v in zip(bars,nr): ax.text(b.get_x()+b.get_width()/2,v+.05,str(v),ha='center',color='#e2e8f0')

ax=axes[0,1]; ax.set_facecolor('#0c1120')
lifts=[r['lift'] for r in rules]
if lifts:
    ax.hist(lifts,bins=max(5,len(lifts)//2),color='#3b82f6',alpha=0.85,edgecolor='#0c1120')
    ax.axvline(np.mean(lifts),color='#06b6d4',linestyle='--',label=f'Mean {np.mean(lifts):.2f}')
    ax.legend(labelcolor='#e2e8f0',framealpha=0)
ax.set_title('Lift Distribution',color='#e2e8f0',pad=10); ax.tick_params(colors='#64748b')
for s in ['top','right']: ax.spines[s].set_visible(False)
for s in ['bottom','left']: ax.spines[s].set_color('#1e293b')

ax=axes[1,0]; ax.set_facecolor('#0c1120')
tp=df['product'].value_counts().head(10)
ax.barh(tp.index[::-1],tp.values[::-1],color=[clrs[i%len(clrs)] for i in range(10)],alpha=0.85)
ax.set_title('Top 10 Products',color='#e2e8f0',pad=10); ax.tick_params(colors='#64748b')
for s in ['top','right']: ax.spines[s].set_visible(False)
for s in ['bottom','left']: ax.spines[s].set_color('#1e293b')

ax=axes[1,1]; ax.set_facecolor('#0c1120')
sc=customer_segs['segment'].value_counts()
wedges,texts,autos=ax.pie(sc.values,labels=sc.index,autopct='%1.1f%%',
    colors=['#3b82f6','#06b6d4','#8b5cf6','#f59e0b'],
    textprops={'color':'#e2e8f0','fontsize':9},wedgeprops={'edgecolor':'#050810','linewidth':2})
for a in autos: a.set_color('#e2e8f0')
ax.set_title('Customer Segments',color='#e2e8f0',pad=10)

plt.suptitle('BasketAI Dashboard',color='#e2e8f0',fontsize=15,fontweight='bold',y=1.01)
plt.tight_layout()
save_path=os.path.join(ROOT,'analysis_dashboard.png')
plt.savefig(save_path,dpi=150,bbox_inches='tight',facecolor='#050810')
plt.show()
print(f'Chart saved → {save_path}')

## 🚀 Cell 11 — Start FastAPI Backend Server

In [ ]:
import subprocess, threading, time, urllib.request

def server_up():
    try: urllib.request.urlopen('http://localhost:8000/',timeout=2); return True
    except: return False

if server_up():
    print('✅ Server already running on http://localhost:8000')
elif not os.path.exists(APP_PY):
    print(f'❌ app.py not found at: {APP_PY}')
else:
    app_dir = os.path.dirname(os.path.abspath(APP_PY))
    print(f'Starting server from: {app_dir}')

    # On Windows use CREATE_NO_WINDOW so no extra console pops up
    kwargs = {}
    if os.name == 'nt':
        kwargs['creationflags'] = subprocess.CREATE_NO_WINDOW

    proc = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'app:app',
         '--host', '0.0.0.0', '--port', '8000'],
        cwd=app_dir,          # ← KEY FIX: run from the folder that contains app.py
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        **kwargs
    )

    print('Waiting for server', end='')
    for _ in range(25):
        time.sleep(1); print('.', end='', flush=True)
        if server_up(): break
    print()

    if server_up():
        print('\n✅ Server running!')
    else:
        try:
            out, _ = proc.communicate(timeout=2)
            print('\nServer output:\n', out.decode()[:1500])
        except subprocess.TimeoutExpired:
            print('\n⚠️  Still starting — wait a few seconds then run Cell 12')

print('\n📖 API docs: http://localhost:8000/docs')

## 🎨 Cell 12 — Verify API & Open Frontend

In [ ]:
import json, urllib.request
from IPython.display import HTML, display

def api_get(path):
    try:
        r=urllib.request.urlopen(f'http://localhost:8000{path}',timeout=5)
        return json.loads(r.read())
    except Exception as e: return {'error':str(e)}

def api_post(path,data):
    try:
        payload=json.dumps(data).encode()
        req=urllib.request.Request(f'http://localhost:8000{path}',data=payload,
                                   headers={'Content-Type':'application/json'},method='POST')
        return json.loads(urllib.request.urlopen(req,timeout=10).read())
    except Exception as e: return {'error':str(e)}

status  = api_get('/api/status')
recs    = api_post('/api/recommend',{'cart':['Laptop','Mouse'],'n_recommendations':3})

print(f'Server alive  : {"✅" if "error" not in status else "❌ "+status.get("error","")}')
print(f'Model ready   : {"✅" if status.get("training_status")=="ready" else "⚠️  wait 10s and rerun"}')
if 'error' not in recs:
    print(f'\nUpsell  : {[r["product"] for r in recs.get("upsell",[])[:3]]}')
    print(f'CrossSell: {[r["product"] for r in recs.get("cross_sell",[])[:3]]}')

# Locate frontend
FRONTEND = next(
    (os.path.abspath(p) for p in [
        os.path.join(ROOT,'frontend','index.html'),
        os.path.join(ROOT,'..','frontend','index.html'),
        os.path.join(os.getcwd(),'frontend','index.html')
    ] if os.path.exists(p)),
    'frontend/index.html (not found)'
)

display(HTML(f"""
<div style="margin-top:20px;background:linear-gradient(135deg,#0c1120,#111827);
  border:1px solid rgba(59,130,246,0.35);border-radius:16px;padding:24px;
  font-family:sans-serif;color:#e2e8f0;max-width:640px;">
  <h2 style="margin:0 0 12px;background:linear-gradient(135deg,#3b82f6,#06b6d4);
    -webkit-background-clip:text;-webkit-text-fill-color:transparent;">◈ BasketAI Ready</h2>
  <div style="display:flex;flex-direction:column;gap:10px;">
    <a href='http://localhost:8000/docs' target='_blank'
       style='background:rgba(59,130,246,0.18);border:1px solid rgba(59,130,246,0.4);
       border-radius:10px;padding:12px 18px;color:#60a5fa;text-decoration:none;font-weight:600;'>
       🔌 API Docs → http://localhost:8000/docs</a>
    <div style='background:rgba(16,185,129,0.1);border:1px solid rgba(16,185,129,0.35);
      border-radius:10px;padding:12px 18px;color:#34d399;font-weight:600;'>
      🎨 Open in browser:<br/>
      <code style='font-size:0.82rem;background:rgba(0,0,0,0.3);padding:2px 6px;border-radius:4px;'>{FRONTEND}</code>
    </div>
    <div style='background:rgba(139,92,246,0.1);border:1px solid rgba(139,92,246,0.3);
      border-radius:10px;padding:10px 18px;color:#a78bfa;font-size:0.85rem;'>
      💡 In JupyterLab right-click <b>frontend/index.html</b> → Open With → HTML Viewer
    </div>
  </div>
</div>
"""))